# track_vis tutorial

This notebook teaches `track_vis` by building up event displays one piece at a time. Every display below is
**live**: drag to rotate, scroll to zoom, right-drag to pan, and hover over a track, marker or voxel to read its
values. Each code cell is short, so change a number and re-run the cell to see what it does. The **Try it** notes
suggest things to change.

All the data here is synthetic (made with NumPy in the notebook), so the tutorial runs anywhere without SNO+ files.

Run it with

```bash
uv sync --group dev
uv run jupyter lab examples/tutorial.ipynb
```

Contents:

1. Your first display: points
2. Lines
3. Truth tracks and vertices (the columnar schema)
4. Binning deposits into voxels
5. Voxel images: `add_volume` and the rendering controls
6. Image and truth side by side
7. Paging through events with `DisplayMenu`
8. Size, themes and slides
9. Saving: standalone pages
10. Under the hood: the `Scene`

In [ ]:
import awkward as ak
import numpy as np
from IPython.display import HTML

from track_vis import DisplayMenu, EventDisplay

rng = np.random.default_rng(1)

## 1. Your first display: points

An `EventDisplay` is an empty 3D view. You add **layers** to it with `add_*` methods, and it draws itself when it
is the last line of a cell. Positions are always in **mm**.

`add_points` draws round markers. Here are 200 points on a helix, coloured by their height `z`. Passing
`colorbar_title` adds a colour bar.

In [ ]:
s = np.linspace(0, 4 * np.pi, 200)
helix = np.stack([300 * np.cos(s), 300 * np.sin(s), 40 * s], axis=1)  # (N, 3) array of x, y, z in mm

disp = EventDisplay(title="A helix of points", height=450)
disp.add_points(helix, "helix", values=helix[:, 2], colorbar_title="z (mm)", size=8)
disp

Things to try in the view above:

- **Drag** to rotate, **scroll** to zoom, **right-drag** (or shift-drag) to pan.
- Click **Options** (top-left) to show the controls. The layer list there doubles as the legend: untick a layer
  to hide it.
- **Click the colour bar** to pick another colour scale, or reverse it.
- The buttons in the bottom-right corner switch to a dark background, reset the camera and go full screen.

**Try it:** change `values=helix[:, 2]` to `values=s` (the helix parameter), or drop `values` and pass
`color="#d62728"` for one colour.

## 2. Lines

`add_line` draws a polyline through an `(N, 3)` array of points. Lines can be one colour, dashed, or coloured
along their length by per-point `values`. Every `add_*` method returns the display, so calls can be chained in
brackets like this:

In [ ]:
t = np.linspace(0, 1, 50)
straight = np.stack([-500 + 1000 * t, 0 * t, 0 * t], axis=1)
wiggly = np.stack([-500 + 1000 * t, 150 * np.sin(8 * t), 200 + 0 * t], axis=1)
curve = np.stack([-500 + 1000 * t, -200 + 0 * t, 300 * t**2], axis=1)

(
    EventDisplay(title="Three kinds of line", height=450)
    .add_line(straight, "solid, one colour", color="#2ca02c", width=4)
    .add_line(wiggly, "dashed", color="#1f77b4", dashed=True, dash=40)
    .add_line(curve, "coloured by values", values=1000 * t, colorbar_title="Distance along (mm)")
)

A list of `(N, 3)` arrays makes several polylines in **one** layer (one entry in the legend). With `hover_text`,
hovering over a point shows your own text, one string per point.

In [ ]:
rays = [np.outer(np.linspace(0, 400, 10), d) for d in rng.normal(size=(12, 3))]
labels = [f"ray {i}, point {j}" for i in range(len(rays)) for j in range(10)]

EventDisplay(title="12 rays, one layer", height=400).add_line(rays, "rays", color="#9467bd", hover_text=labels)

## 3. Truth tracks and vertices

`add_tracks` and `add_vertices` take MC truth in the **columnar** layout of the SNO+ `imaging_reco` ntuple
(what `track_vis.load_event` returns):

| Column | Shape | Meaning |
| --- | --- | --- |
| `pdg` | one entry per track | PDG code, e.g. 11 = e-, 22 = gamma, 2212 = proton, -13 = mu+ |
| `pos_x`, `pos_y`, `pos_z` | `[track][step]` (jagged) | step positions (mm) |
| `time` | `[track][step]` | step times (ns) |
| `deposited_energy` | `[track][step]` | energy deposited at each step (MeV) |

"Jagged" means each track can have a different number of steps. Plain Python lists of lists work; `awkward`
arrays (what `uproot` gives you) work too.

The function below fakes one track as a wobbly random walk, so we can build a small event: an electron, a
proton and a gamma leaving a common vertex.

In [ ]:
def fake_track(direction, n_steps, step_mm, t0=0.0, dedx=0.1, wobble=2.0):
    """A straight-ish random walk from the origin: positions (n, 3), times (n,) and deposits (n,)."""
    direction = np.asarray(direction, float) / np.linalg.norm(direction)
    steps = direction * step_mm + rng.normal(0, wobble, (n_steps, 3))
    pos = np.cumsum(steps, axis=0) - steps[0]  # start at the origin
    time = t0 + np.arange(n_steps) * step_mm / 200.0  # ~200 mm/ns
    dep = dedx * rng.uniform(0.5, 1.5, n_steps)
    return pos, time, dep


particles = [  # (pdg, direction, steps, step length, dE per step)
    (11, [1, 0.3, 0.2], 40, 10, 0.05),
    (2212, [-0.5, 1, 0.1], 25, 6, 0.3),
    (22, [-0.2, -0.8, 0.9], 10, 40, 0.0),  # a gamma deposits nothing itself
]
tracks = {"pdg": np.array([p[0] for p in particles])}
for name in ("pos_x", "pos_y", "pos_z", "time", "deposited_energy"):
    tracks[name] = []
for pdg, direction, n, step, dedx in particles:
    pos, time, dep = fake_track(direction, n, step, dedx=dedx)
    tracks["pos_x"].append(pos[:, 0])
    tracks["pos_y"].append(pos[:, 1])
    tracks["pos_z"].append(pos[:, 2])
    tracks["time"].append(time)
    tracks["deposited_energy"].append(dep)
tracks = {k: ak.Array(v) if k != "pdg" else v for k, v in tracks.items()}

vertices = {  # flat arrays, one entry per vertex (the ntuple's mcpdgs, mcxs, ...)
    "pdg": np.array([11, 2212]),
    "x": np.array([0.0, 0.0]),
    "y": np.array([0.0, 0.0]),
    "z": np.array([0.0, 0.0]),
    "t": np.array([0.0, 0.0]),
    "energy": np.array([2.0, 0.8]),  # kinetic energy (MeV)
}
tracks["pos_x"]

Tracks are coloured by **time** over `t_min`..`t_max`, with one layer per particle species. Vertices are markers
coloured on the same scale, so pass both calls the **same** `t_min` and `t_max` and they share one
"Time (ns)" colour bar. Hover over a track step or a vertex to read its time and energy.

In [ ]:
all_t = ak.to_numpy(ak.flatten(tracks["time"]))
t_min, t_max = float(all_t.min()), float(all_t.max())

(
    EventDisplay(title="e- / p / gamma from a common vertex", height=500)
    .add_tracks(**tracks, t_min=t_min, t_max=t_max)
    .add_vertices(**vertices, t_min=t_min, t_max=t_max)
)

With a real ntuple it is one call to load an event, and the same two calls to draw it:

```python
from track_vis import load_event

tracks, vertices = load_event("job_0.ntuple.root", index=0)
disp.add_tracks(**tracks, t_min=t_min, t_max=t_max).add_vertices(**vertices, t_min=t_min, t_max=t_max)
```

## 4. Binning deposits into voxels

`add_deposits` takes the same track columns, sums each step's `deposited_energy` into cubes of side
`voxel_size` (mm), and draws them as a **voxel volume**. Volumes are drawn on the GPU, so every voxel has its
own opacity and nothing flickers while you rotate.

In [ ]:
(
    EventDisplay(title="Truth deposits in 20 mm voxels", height=500)
    .add_deposits(tracks["pos_x"], tracks["pos_y"], tracks["pos_z"], tracks["deposited_energy"], voxel_size=20)
    .add_tracks(**tracks, t_min=t_min, t_max=t_max)
)

**Try it:** change `voxel_size` to 10 or 50. Untick the track layers in **Options** to see the voxels alone.

## 5. Voxel images: `add_volume`

A reconstructed image (e.g. an MLEM image) goes in with `add_volume`, in one of two forms:

- **dense**: a 3D array `grid[ix, iy, iz]` plus `origin`, the min corner (mm) of voxel `[0, 0, 0]`;
- **sparse**: `centers` (an `(N, 3)` array of voxel centres on a regular grid) and their `energies`.

`voxel_size` is needed for both. Here is a fake "muon image": a blurred line segment on a 50 mm grid, with some
noise.

In [ ]:
def blurred_segment(start, end, voxel=50.0, sigma=45.0, noise=0.1):
    """A dense image of a blurred line segment from `start` to `end` (mm). Returns (grid, origin)."""
    start, end = np.asarray(start, float), np.asarray(end, float)
    lo = np.floor((np.minimum(start, end) - 4 * sigma) / voxel) * voxel
    hi = np.ceil((np.maximum(start, end) + 4 * sigma) / voxel) * voxel
    axes = [np.arange(a, b, voxel) + voxel / 2 for a, b in zip(lo, hi)]  # voxel centres along x, y, z
    centres = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1)
    d = end - start
    s = np.clip((centres - start) @ d / (d @ d), 0, 1)  # position along the segment, 0..1
    r = np.linalg.norm(centres - (start + s[..., None] * d), axis=-1)  # distance from the segment
    image = np.exp(-0.5 * (r / sigma) ** 2) * (0.6 + 0.4 * s)  # brighter towards the end
    image *= rng.gamma(1 / noise**2, noise**2, image.shape)  # multiplicative noise
    image[image < 0.03 * image.max()] = 0
    return image, lo


mu_start, mu_end = np.array([-250.0, -100, -150]), np.array([250.0, 150, 200])
image, origin = blurred_segment(mu_start, mu_end)
print("grid shape", image.shape, "origin", origin)

EventDisplay(title="A fake 50 mm muon image", height=500).add_volume(image, origin=origin, voxel_size=50)

The same image in sparse form gives the same picture. This is handy when your reconstruction stores a list of
non-empty voxels:

In [ ]:
ix, iy, iz = np.nonzero(image)
centers = origin + (np.stack([ix, iy, iz], axis=1) + 0.5) * 50  # voxel centres (mm)
energies = image[ix, iy, iz]

EventDisplay(title="Same image, from sparse voxels", height=400).add_volume(
    centers=centers, energies=energies, voxel_size=50
)

### The rendering controls

Open **Options** in a display with a volume to get three sliders and two switches. Their starting values are
`EventDisplay` arguments:

| Argument | Slider / switch | What it does |
| --- | --- | --- |
| `threshold=0.05` | threshold | voxels below this fraction of `e_max` are fully transparent |
| `density=2.0` | density | how opaque a bright voxel is; higher hides what is behind |
| `gamma=2.0` | gamma | opacity curve; above 1 fades faint voxels, below 1 brings them up |
| `smooth=False` | smooth | trilinear interpolation instead of sharp cubes |
| | MIP | maximum-intensity projection: each pixel shows the brightest voxel along the ray |

The view below starts smooth, with a low threshold and gamma so the faint halo shows. Move the sliders to
compare.

In [ ]:
EventDisplay(title="Smooth, faint voxels shown", height=500, threshold=0.01, gamma=1.0, smooth=True).add_volume(
    image, origin=origin, voxel_size=50
)

**Comparing images.** The colour and opacity scale runs from 0 to `e_max`, which defaults to each image's
own maximum. To compare two events on one scale, pass the same `e_max` to both. `colorscale` picks the colour
scale (plasma, viridis, inferno, magma, cividis, turbo), and `name` and `colorbar_title` set the labels.

In [ ]:
EventDisplay(title="Fixed e_max, viridis", height=400).add_volume(
    image,
    origin=origin,
    voxel_size=50,
    e_max=2 * image.max(),
    colorscale="viridis",
    name="MLEM image",
    colorbar_title="Image (MeV)",
)

## 6. Image and truth side by side

Layers mix freely, so the simplest comparison is one display with the image, the truth track and a fitted
axis (`add_line` with `dashed=True`). Untick layers to compare.

In [ ]:
n = 60
s = np.linspace(0, 1, n)[:, None]
mu_path = mu_start + s * (mu_end - mu_start) + rng.normal(0, 3, (n, 3)).cumsum(axis=0) * 0.3
mu_tracks = {
    "pdg": np.array([-13]),
    "pos_x": ak.Array([mu_path[:, 0]]),
    "pos_y": ak.Array([mu_path[:, 1]]),
    "pos_z": ak.Array([mu_path[:, 2]]),
    "time": ak.Array([np.linspace(0, 2.2, n)]),
    "deposited_energy": ak.Array([np.full(n, 1.0)]),
}
fit = np.stack([mu_start + [10, -5, 0], mu_end + [-5, 10, 5]])

muon = (
    EventDisplay(title="Image, truth and fitted axis", height=500)
    .add_volume(image, origin=origin, voxel_size=50, name="MLEM image")
    .add_tracks(**mu_tracks)
    .add_line(fit, "fitted axis", color="#2563eb", dashed=True, width=2)
)
muon

To put **two displays next to each other**, wrap their HTML (`to_html()`) in a flex box. Give both the same
`bounds` (the view box, `[[x_min, y_min, z_min], [x_max, y_max, z_max]]` in mm) so they are drawn at the same
scale.

In [ ]:
def side_by_side(*displays, gap=8):
    """Show displays in one row; each takes an equal share of the width."""
    cells = "".join(f'<div style="flex:1;min-width:0">{d.to_html()}</div>' for d in displays)
    return HTML(f'<div style="display:flex;gap:{gap}px">{cells}</div>')


box = [[-400, -300, -350], [400, 350, 400]]
side_by_side(
    EventDisplay(title="reconstructed image", height=400, bounds=box).add_volume(image, origin=origin, voxel_size=50),
    EventDisplay(title="truth", height=400, bounds=box).add_tracks(**mu_tracks),
)

## 7. Paging through events with `DisplayMenu`

A page with many 3D views gets slow (browsers allow about 16 live WebGL views per page). `DisplayMenu` puts
several displays in **one** view with a drop-down and previous/next buttons, so you can flick through events.
Only the display on show is drawn.

In [ ]:
def fake_muon_event(seed):
    """An EventDisplay of a random fake muon image with its truth track."""
    r = np.random.default_rng(seed)
    a = r.uniform(-300, 300, 3)
    direction = r.normal(size=3)
    b = a + direction / np.linalg.norm(direction) * r.uniform(400, 700)  # 0.4-0.7 m long
    img, org = blurred_segment(a, b)
    path = np.linspace(a, b, 30)
    truth = {
        "pdg": [-13],
        "pos_x": [path[:, 0]],
        "pos_y": [path[:, 1]],
        "pos_z": [path[:, 2]],
        "time": [np.linspace(0, 2, 30)],
        "deposited_energy": [np.ones(30)],
    }
    return (
        EventDisplay(title=f"fake muon, seed {seed}", height=450, bounds=[[-700] * 3, [700] * 3])
        .add_volume(img, origin=org, voxel_size=50, name="image")
        .add_tracks(**truth)
    )


DisplayMenu([fake_muon_event(seed) for seed in range(5)], labels=[f"event {seed}" for seed in range(5)])

The fixed `bounds` keep the axes and camera the same as you page, so the events are easy to compare. Leave it
out to fit each event's own box.

## 8. Size, themes and slides

### Size

- `height` is the view height in pixels (default 600).
- `width` is the **maximum** width in pixels (default 900). The view shrinks to fit narrower pages, so it
  works on a phone or in a narrow column. Pass a large `width` to fill a wide page.

For an existing display, change `disp.scene.height` and `disp.scene.width` before showing it again.

In [ ]:
EventDisplay(title="Short and wide", width=1400, height=300).add_volume(image, origin=origin, voxel_size=50)

### Themes

`theme="dark"` starts the view on a dark background, where faint voxels and the bright end of the colour
scales stand out. The moon/sun button in the view switches either way.

In [ ]:
EventDisplay(title="Dark theme", height=450, theme="dark").add_volume(image, origin=origin, voxel_size=50).add_tracks(
    **mu_tracks
)

### In Quarto slides (revealjs)

A display renders in a Quarto document or revealjs deck exactly as in this notebook: make it the last line of a
`{python}` cell. For slides, two things help:

1. Set `height` to fit under the slide title. With the template's 1280 x 720 slides, about **480 px** fills a
   slide under a one-line title and about **420 px** fits one of two columns.
2. Use `to_html(lib="inline")` so three.js is embedded in the deck, which then works offline (no internet at the
   conference) and as a single file.

A small helper does both (this is `show` in the project's slide template):

```python
from IPython.display import HTML


def show(display, height=480):
    """Draw an EventDisplay or DisplayMenu in a slide, with three.js embedded."""
    for d in getattr(display, "displays", [display]):  # a DisplayMenu holds several
        d.scene.height = height
    return HTML(display.to_html(lib="inline"))
```

and a slide with two columns:

````markdown
## Image against truth

:::: {.columns}
::: {.column width="50%"}
```{python}
show(image_display, 420)
```
:::
::: {.column width="50%"}
```{python}
show(truth_display, 420)
```
:::
::::
````

Print to PDF from Chrome with `talk.html?print-pdf`: the 3D views are captured as they are on screen. For
PowerPoint or Beamer, take a PNG of a `write_html` page (next section) with a headless browser.

## 9. Saving: standalone pages

`write_html` writes a page you can open in any browser, e-mail, or attach to a logbook. By default three.js is
embedded (`lib="inline"`), so the page works offline. `DisplayMenu` has the same method.

In [ ]:
muon.write_html("muon_event.html")
print("wrote muon_event.html")

`to_html()` returns the same display as an HTML fragment, for pasting into your own page:

| Call | three.js | Use it for |
| --- | --- | --- |
| `disp` (last line of a cell) | from the jsDelivr CDN | notebooks and Quarto pages |
| `disp.to_html(lib="cdn")` | from the CDN | your own HTML, smallest file |
| `disp.to_html(lib="inline")` | embedded (~600 kB) | slides and pages that must work offline |
| `disp.write_html(path)` | embedded | a standalone file |

## 10. Under the hood: the `Scene`

An `EventDisplay` only builds a `Scene`: plain Python dataclasses of NumPy arrays, one per layer. Nothing in it
is JavaScript. `track_vis/browser.py` turns the scene into JSON and hands it to the renderer in
`track_vis/js/event_display.js`, so you can do everything from Python and never touch the JavaScript.

In [ ]:
scene = muon.scene
print("volumes:", [v.name for v in scene.volumes])
print("lines:  ", [ln.name for ln in scene.lines])
print("colour bars:", [b.title for b in scene.colorbars])
print("image grid:", scene.volumes[0].grid.shape, "voxel size", scene.volumes[0].voxel_size, "mm")

That is the whole API. The full list of methods and arguments is in the README, and every method has a
docstring: try `EventDisplay.add_volume?` in a cell.